# 第 11 课 Notebook：FDE 垂直切片：从问题发现到可交付工作流

**核心问题：怎样在部分失败时保持租户、版本、幂等、Outbox 与补偿的一致性？**

本 Notebook 先用本地固定夹具运行正常路径，再主动触发一条失败路径。所有副作用均为内存、临时目录或仿真；不需要 API Key、外网、真实邮箱、ERP、支付或机器人。代码字段与日志使用英文，讲解使用中文。

## 阅读路线

1. 先预测终态；2. 认识本课核心对象；3. 运行正常路径；4. 触发失败；5. 写出不变量；6. 把局部机制接到 Project。不要先背框架名称，先沿状态变化解释“事情为什么发生”。

## 1. 先预测

在运行前写下你的判断：如果系统只检查最终文字，不读取环境终态，本课案例最可能漏掉什么？课程核心问句是：**怎样在部分失败时保持租户、版本、幂等、Outbox 与补偿的一致性？**

In [ ]:
lesson_id = 11
essential_question = '怎样在部分失败时保持租户、版本、幂等、Outbox 与补偿的一致性？'
prediction = "write your prediction before running the next cell"
print({"lesson_id": lesson_id, "question": essential_question, "prediction": prediction})

## 2. 核心对象：`ServiceStore`

先看对象的边界：它接收什么、保存什么状态、返回什么终态、哪种失败必须由外部程序或人处理。下面先使用本课公开的教学核心，观察接口和状态；完整实现留在项目 starter 的任务边界中。

In [ ]:
print({"teaching_boundary": "本页只观察公开教学核心的输入、状态和输出；不读取教师实现源码。"})

## 3. 正常路径：先看真实结果

运行时先找输入、关键中间状态和终态。若输出很长，不要逐行朗读；只标出本课负责的新对象。

In [ ]:
from agent_course.courseware.public_examples import *
service_outcome, service_case, service_platform, service_mail = await run_service_demo()
print(service_outcome.model_dump(mode='json'))
print(service_case.model_dump(mode='json'))


## 4. 解释正常路径

用四句话复述：输入是什么；谁拥有状态；程序允许了哪个动作；哪个环境结果证明任务完成。若只能复述 Agent 的自然语言，就还没有解释闭环。

In [ ]:
normal_path_check = {
    "input_visible": True,
    "state_visible": True,
    "terminal_outcome_visible": True,
    "real_external_side_effect": False,
}
required_checks = (
    "input_visible",
    "state_visible",
    "terminal_outcome_visible",
)
assert all(normal_path_check[key] for key in required_checks)
assert normal_path_check["real_external_side_effect"] is False
print(normal_path_check)

## 5. 失败路径：能力必须和边界一起出现

现在保持任务与实现不变，只改变一个关键条件。观察失败发生在提议、授权、执行、反馈还是终态判断。

In [ ]:
from agent_course.courseware.public_examples import Channel, InboundRequest, ServiceCase, ServiceStore, StoreViolation

transaction_store = ServiceStore()
case = transaction_store.create_case(ServiceCase('case-version', InboundRequest('request-version', 'tenant-au', Channel.EMAIL, 'customer@example.test', 'Version check', 'Please record this case.')))
updated = transaction_store.save_case(case, expected_version=case.version)
try:
    transaction_store.save_case(case, expected_version=case.version)
except StoreViolation as error:
    stale_write_reason = str(error)
    print({'stale_write_blocked': True, 'reason': stale_write_reason, 'current_version': updated.version})
else:
    raise AssertionError('stale write should be blocked')


## 6. 把正常与失败路径画在一起

下面的图直接读取本次运行产生的状态、结果与停止原因。先沿图复述“输入—动作—环境结果—终态”，再回到代码定位每一项由哪一行产生。图不是额外的装饰层。

In [ ]:
from IPython.display import HTML, display

from agent_course.courseware.project_visuals import (
    VisualStep,
    render_swimlane,
)

transaction_steps = (
    VisualStep(
        "Case version",
        f"version={service_case.version}",
        "passed",
        lane="Store",
    ),
    VisualStep(
        "Supervisor approval",
        service_case.approval.approval_id,
        "passed",
        lane="Approval",
    ),
    VisualStep(
        "Create ticket",
        f"effects={len(service_platform.commands)}",
        "passed",
        lane="Platform",
    ),
    VisualStep(
        "Send test mail",
        f"receipts={len(service_mail.messages)}",
        "passed",
        lane="Mail",
    ),
    VisualStep(
        "Stale write",
        stale_write_reason,
        "blocked",
        lane="Store",
    ),
    VisualStep(
        "Business terminal state",
        service_outcome.stop_reason.value,
        "passed",
        lane="Runtime",
    ),
)
display(
    HTML(
        render_swimlane(
            title="企业事务：局部成功必须收敛为一个业务终态",
            question="如果邮件失败，哪一个已经成功的动作不能重复？",
            lanes=(
                "Store",
                "Approval",
                "Platform",
                "Mail",
                "Runtime",
            ),
            steps=transaction_steps,
            takeaway="版本、批准、幂等 receipt 与 Outbox 共同决定如何恢复。",
        )
    )
)

## 6.1 不变量

不变量是无论模型怎样表达都不能被绕过的规则。它应由普通代码、存储约束、沙箱或独立监督器执行。

In [ ]:
invariants = (
    "terminal outcome is explicit",
    "failure is represented as data",
    "side effects stay inside the course sandbox",
    "a model statement cannot override a deterministic boundary",
)
assert len(invariants) == 4
print(invariants)

## 7. 连接到 Project

ServiceOps 事务里程碑：乐观并发、Outbox、幂等副作用、失败重试与对账。 本节代码不是另一本工具说明书，而是项目验收中的一个可重复证据。

## 8. 学生练习

完成下面的小函数。允许使用 Agent 帮助阅读报错和补充测试，但你必须能逐行解释判断条件，并亲自确认失败路径。

In [ ]:
def may_retry(idempotency_key: str, effect_exists: bool) -> str:
    # TODO: 已存在副作用时返回旧 receipt，不再次执行。
    raise NotImplementedError

assert may_retry("case-1:ticket", True) == "return_existing_receipt"
assert may_retry("case-1:ticket", False) == "execute:case-1:ticket"

## 9. 展示与验收

展示时只讲五件事：任务输入、关键状态、正常终态、失败终态、一个不能被模型绕过的不变量。练习时改变一项夹具，解释结果为什么改变。

## 10. 小结

回答开场问题：**怎样在部分失败时保持租户、版本、幂等、Outbox 与补偿的一致性？**

请用“输入 → 状态 → 动作 → 环境结果 → 停止原因”写一段不超过 120 字的复盘，并把失败断言迁移到 Project 测试。